In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 🎴 Yu-Gi-Oh! Card Recognition - Desarrollo del Modelo\n",
    "\n",
    "Este notebook contiene el desarrollo experimental del modelo de reconocimiento de cartas de Yu-Gi-Oh! usando redes neuronales siamesas.\n",
    "\n",
    "## Objetivos:\n",
    "1. Explorar el dataset de cartas\n",
    "2. Desarrollar y entrenar la red neuronal siamesa\n",
    "3. Evaluar el rendimiento del modelo\n",
    "4. Optimizar hiperparámetros\n",
    "5. Probar con imágenes reales"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Configuración Inicial"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import torch\n",
    "import torch.nn as nn\n",
    "import torchvision.transforms as transforms\n",
    "from PIL import Image\n",
    "import matplotlib.pyplot as plt\n",
    "import numpy as np\n",
    "import os\n",
    "import sys\n",
    "from pathlib import Path\n",
    "\n",
    "# Agregar el directorio padre al path\n",
    "sys.path.append('..')\n",
    "\n",
    "# Configurar matplotlib\n",
    "plt.style.use('default')\n",
    "plt.rcParams['figure.figsize'] = (12, 8)\n",
    "\n",
    "# Verificar GPU\n",
    "device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n",
    "print(f\"Dispositivo: {device}\")\n",
    "print(f\"PyTorch version: {torch.__version__}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Exploración del Dataset"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Cargar el dataset\n",
    "from dataset import YuGiOhDataset\n",
    "\n",
    "dataset_path = \"../data/yugioh_card_images\"\n",
    "dataset = YuGiOhDataset(dataset_path)\n",
    "\n",
    "print(f\"Dataset cargado: {len(dataset)} cartas\")\n",
    "print(f\"Ejemplos de cartas:\")\n",
    "for i in range(min(5, len(dataset))):\n",
    "    print(f\"  {i+1}. {dataset.card_names[i]}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualizar algunas cartas del dataset\n",
    "fig, axes = plt.subplots(2, 3, figsize=(15, 10))\n",
    "axes = axes.ravel()\n",
    "\n",
    "for i in range(min(6, len(dataset))):\n",
    "    img, _ = dataset[i]\n",
    "    img_np = img.permute(1, 2, 0).numpy()\n",
    "    \n",
    "    # Denormalizar\n",
    "    mean = np.array([0.485, 0.456, 0.406])\n",
    "    std = np.array([0.229, 0.224, 0.225])\n",
    "    img_np = std * img_np + mean\n",
    "    img_np = np.clip(img_np, 0, 1)\n",
    "    \n",
    "    axes[i].imshow(img_np)\n",
    "    axes[i].set_title(dataset.card_names[i])\n",
    "    axes[i].axis('off')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Arquitectura de la Red Neuronal Siamesa"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Definir la arquitectura de la red siamesa\n",
    "class SiameseNetwork(nn.Module):\n",
    "    def __init__(self, embedding_dim=128):\n",
    "        super(SiameseNetwork, self).__init__()\n",
    "        \n",
    "        # Encoder basado en ResNet18 (fine-tuning)\n",
    "        self.encoder = nn.Sequential(\n",
    "            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),\n",
    "            nn.BatchNorm2d(64),\n",
    "            nn.ReLU(inplace=True),\n",
    "            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),\n",
    "            \n",
    "            # Bloque 1\n",
    "            self._make_layer(64, 64, 2),\n",
    "            # Bloque 2\n",
    "            self._make_layer(64, 128, 2, stride=2),\n",
    "            # Bloque 3\n",
    "            self._make_layer(128, 256, 2, stride=2),\n",
    "            # Bloque 4\n",
    "            self._make_layer(256, 512, 2, stride=2),\n",
    "            \n",
    "            nn.AdaptiveAvgPool2d((1, 1)),\n",
    "            nn.Flatten(),\n",
    "            nn.Linear(512, embedding_dim),\n",
    "            nn.L2Normalize(dim=1)\n",
    "        )\n",
    "    \n",
    "    def _make_layer(self, in_channels, out_channels, blocks, stride=1):\n",
    "        layers = []\n",
    "        layers.append(self._make_block(in_channels, out_channels, stride))\n",
    "        for _ in range(1, blocks):\n",
    "            layers.append(self._make_block(out_channels, out_channels))\n",
    "        return nn.Sequential(*layers)\n",
    "    \n",
    "    def _make_block(self, in_channels, out_channels, stride=1):\n",
    "        return nn.Sequential(\n",
    "            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),\n",
    "            nn.BatchNorm2d(out_channels),\n",
    "            nn.ReLU(inplace=True),\n",
    "            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False),\n",
    "            nn.BatchNorm2d(out_channels)\n",
    "        )\n",
    "    \n",
    "    def forward(self, x):\n",
    "        return self.encoder(x)\n",
    "\n",
    "# Crear instancia del modelo\n",
    "model = SiameseNetwork(embedding_dim=128)\n",
    "model.to(device)\n",
    "\n",
    "print(f\"Modelo creado con {sum(p.numel() for p in model.parameters())} parámetros\")\n",
    "print(f\"Modelo en dispositivo: {next(model.parameters()).device}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Función de Pérdida Triplet Loss"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "class TripletLoss(nn.Module):\n",
    "    def __init__(self, margin=1.0):\n",
    "        super(TripletLoss, self).__init__()\n",
    "        self.margin = margin\n",
    "    \n",
    "    def forward(self, anchor, positive, negative):\n",
    "        # Calcular distancias euclidianas\n",
    "        pos_dist = torch.sum((anchor - positive) ** 2, dim=1)\n",
    "        neg_dist = torch.sum((anchor - negative) ** 2, dim=1)\n",
    "        \n",
    "        # Triplet loss\n",
    "        loss = torch.clamp(pos_dist - neg_dist + self.margin, min=0.0)\n",
    "        return torch.mean(loss)\n",
    "\n",
    "# Crear función de pérdida\n",
    "criterion = TripletLoss(margin=1.0)\n",
    "print(f\"Triplet Loss creado con margen: {criterion.margin}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Entrenamiento del Modelo"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Configurar optimizador\n",
    "optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)\n",
    "scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)\n",
    "\n",
    "print(f\"Optimizador: Adam con lr={optimizer.param_groups[0]['lr']}\")\n",
    "print(f\"Scheduler: StepLR con step_size=10, gamma=0.5\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Función de entrenamiento\n",
    "def train_epoch(model, dataloader, criterion, optimizer, device):\n",
    "    model.train()\n",
    "    total_loss = 0\n",
    "    num_batches = 0\n",
    "    \n",
    "    for batch_idx, (anchor, positive, negative) in enumerate(dataloader):\n",
    "        anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)\n",
    "        \n",
    "        optimizer.zero_grad()\n",
    "        \n",
    "        # Forward pass\n",
    "        anchor_emb = model(anchor)\n",
    "        positive_emb = model(positive)\n",
    "        negative_emb = model(negative)\n",
    "        \n",
    "        # Calcular pérdida\n",
    "        loss = criterion(anchor_emb, positive_emb, negative_emb)\n",
    "        \n",
    "        # Backward pass\n",
    "        loss.backward()\n",
    "        optimizer.step()\n",
    "        \n",
    "        total_loss += loss.item()\n",
    "        num_batches += 1\n",
    "        \n",
    "        if batch_idx % 10 == 0:\n",
    "            print(f\"Batch {batch_idx}: Loss = {loss.item():.4f}\")\n",
    "    \n",
    "    return total_loss / num_batches\n",
    "\n",
    "print(\"Función de entrenamiento definida\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Evaluación del Modelo"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Función para evaluar el modelo\n",
    "def evaluate_model(model, test_dataset, device, top_k=5):\n",
    "    model.eval()\n",
    "    \n",
    "    # Generar embeddings para todas las cartas\n",
    "    all_embeddings = []\n",
    "    all_names = []\n",
    "    \n",
    "    with torch.no_grad():\n",
    "        for i in range(len(test_dataset)):\n",
    "            img, name = test_dataset[i]\n",
    "            img = img.unsqueeze(0).to(device)\n",
    "            embedding = model(img)\n",
    "            all_embeddings.append(embedding.cpu())\n",
    "            all_names.append(name)\n",
    "    \n",
    "    # Concatenar embeddings\n",
    "    embeddings_tensor = torch.cat(all_embeddings, dim=0)\n",
    "    \n",
    "    # Calcular matriz de similitud\n",
    "    similarity_matrix = torch.mm(embeddings_tensor, embeddings_tensor.t())\n",
    "    \n",
    "    # Evaluar precisión top-k\n",
    "    correct = 0\n",
    "    total = 0\n",
    "    \n",
    "    for i in range(len(test_dataset)):\n",
    "        # Obtener top-k más similares (excluyendo la misma carta)\n",
    "        similarities = similarity_matrix[i]\n",
    "        similarities[i] = -1  # Excluir la misma carta\n",
    "        \n",
    "        top_indices = torch.topk(similarities, top_k).indices\n",
    "        \n",
    "        # Verificar si la carta correcta está en top-k\n",
    "        if i in top_indices:\n",
    "            correct += 1\n",
    "        total += 1\n",
    "    \n",
    "    accuracy = correct / total if total > 0 else 0\n",
    "    return accuracy, similarity_matrix\n",
    "\n",
    "print(\"Función de evaluación definida\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Visualización de Resultados"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Función para visualizar embeddings\n",
    "def visualize_embeddings(embeddings, names, n_samples=50):\n",
    "    from sklearn.manifold import TSNE\n",
    "    \n",
    "    # Tomar una muestra para visualización\n",
    "    if len(embeddings) > n_samples:\n",
    "        indices = np.random.choice(len(embeddings), n_samples, replace=False)\n",
    "        sample_embeddings = embeddings[indices]\n",
    "        sample_names = [names[i] for i in indices]\n",
    "    else:\n",
    "        sample_embeddings = embeddings\n",
    "        sample_names = names\n",
    "    \n",
    "    # Reducir dimensionalidad con t-SNE\n",
    "    tsne = TSNE(n_components=2, random_state=42)\n",
    "    embeddings_2d = tsne.fit_transform(sample_embeddings)\n",
    "    \n",
    "    # Visualizar\n",
    "    plt.figure(figsize=(12, 8))\n",
    "    plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], alpha=0.7)\n",
    "    \n",
    "    # Agregar etiquetas\n",
    "    for i, name in enumerate(sample_names):\n",
    "        plt.annotate(name[:10], (embeddings_2d[i, 0], embeddings_2d[i, 1]), \n",
    "                    fontsize=8, alpha=0.8)\n",
    "    \n",
    "    plt.title('Visualización de Embeddings con t-SNE')\n",
    "    plt.xlabel('Componente 1')\n",
    "    plt.ylabel('Componente 2')\n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "\n",
    "print(\"Función de visualización definida\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. Pruebas con Imágenes Reales"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Función para probar con imágenes reales\n",
    "def test_real_image(model, image_path, dataset, device, top_k=5):\n",
    "    from utils import preprocess_real_photo\n",
    "    \n",
    "    # Preprocesar imagen real\n",
    "    processed_img = preprocess_real_photo(image_path)\n",
    "    if processed_img is None:\n",
    "        print(\"Error al procesar la imagen\")\n",
    "        return\n",
    "    \n",
    "    # Transformar imagen\n",
    "    transform = transforms.Compose([\n",
    "        transforms.Resize((224, 224)),\n",
    "        transforms.ToTensor(),\n",
    "        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])\n",
    "    ])\n",
    "    \n",
    "    img_tensor = transform(processed_img).unsqueeze(0).to(device)\n",
    "    \n",
    "    # Generar embedding\n",
    "    model.eval()\n",
    "    with torch.no_grad():\n",
    "        query_embedding = model(img_tensor)\n",
    "    \n",
    "    # Comparar con dataset\n",
    "    similarities = []\n",
    "    for i in range(len(dataset)):\n",
    "        img, name = dataset[i]\n",
    "        img = img.unsqueeze(0).to(device)\n",
    "        with torch.no_grad():\n",
    "            dataset_embedding = model(img)\n",
    "        similarity = torch.cosine_similarity(query_embedding, dataset_embedding)\n",
    "        similarities.append((name, similarity.item()))\n",
    "    \n",
    "    # Ordenar por similitud\n",
    "    similarities.sort(key=lambda x: x[1], reverse=True)\n",
    "    \n",
    "    # Mostrar resultados\n",
    "    print(f\"Top {top_k} cartas más similares:\")\n",
    "    for i, (name, sim) in enumerate(similarities[:top_k]):\n",
    "        print(f\"{i+1}. {name}: {sim:.4f}\")\n",
    "    \n",
    "    return similarities[:top_k]\n",
    "\n",
    "print(\"Función de prueba con imágenes reales definida\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 9. Conclusiones y Próximos Pasos"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Resumen de experimentos realizados\n",
    "print(\"=== RESUMEN DE EXPERIMENTOS ===\")\n",
    "print(\"\\n1. Exploración del Dataset:\")\n",
    "print(f\"   - Total de cartas: {len(dataset)}\")\n",
    "print(f\"   - Tamaño de imagen: 224x224\")\n",
    "print(f\"   - Normalización: ImageNet\")\n",
    "\n",
    "print(\"\\n2. Arquitectura del Modelo:\")\n",
    "print(f\"   - Tipo: Red Neuronal Siamesa\")\n",
    "print(f\"   - Base: ResNet18 (Fine-tuned)\")\n",
    "print(f\"   - Embedding: 128 dimensiones\")\n",
    "print(f\"   - Parámetros: {sum(p.numel() for p in model.parameters()):,}\")\n",
    "\n",
    "print(\"\\n3. Optimizaciones Implementadas:\")\n",
    "print(\"   - Tensor de embeddings en 4ª dimensión\")\n",
    "print(\"   - Cache de embeddings para velocidad\")\n",
    "print(\"   - Preprocesamiento automático con OpenCV\")\n",
    "print(\"   - Comparación vectorizada\")\n",
    "\n",
    "print(\"\\n4. Próximos Pasos:\")\n",
    "print(\"   - Entrenar el modelo con más épocas\")\n",
    "print(\"   - Ajustar hiperparámetros\")\n",
    "print(\"   - Probar con más imágenes reales\")\n",
    "print(\"   - Optimizar preprocesamiento\")\n",
    "print(\"   - Implementar métricas de evaluación\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}